# Definition of the observable night (ROSP-22)

Supports the ROSP-9 decision on the denominator of system availability. Changing the definition later is a series break.

**Decision:** the observable night of a dayObs is the time between evening and morning twilight, from the moment the Sun reaches altitude **-12 deg** in the evening to the moment it reaches **-12 deg** in the morning (`TWILIGHT_ALT = -12`). It does not depend on the weather, the observatory state or the dome.

The notebook has two parts:

1. **The definition** (next cell). It needs only `rubin_nights` and is computed from the Sun's altitude alone. Example plots of the night length and a calendar are shown for a full year.
2. **Historical comparison.** Other definitions that were considered, built from the EFD-recorded observatory state. They are kept for the record only and are **not valid** ways to calculate the observable night.

This notebook is meant to be run locally at USDF (`lsst` kernel); it is not a Times Square notebook.

- Jira: [ROSP-9](https://rubinobs.atlassian.net/browse/ROSP-9), [ROSP-22](https://rubinobs.atlassian.net/browse/ROSP-22)

In [ ]:
# THE DEFINITION OF THE OBSERVABLE NIGHT
#
# For each dayObs, the observable night runs from the evening to the morning moment at which the Sun is at
# altitude TWILIGHT_ALT (-12 deg). It is computed from the Sun's altitude only, with no EFD or state data.
import numpy as np
import pandas as pd

import rubin_nights.dayobs_utils as rn_dayobs

TWILIGHT_ALT = (
    -12.0
)  # Sun altitude (deg) at the start (evening) and end (morning) of the observable night


def observable_night(day_obs):
    """Start and end of the observable night of one dayObs, as astropy Times."""
    start, end = rn_dayobs.day_obs_sunset_sunrise(day_obs, TWILIGHT_ALT)
    return start, end


example_dayObs = 20260629

example_observable_night = observable_night(example_dayObs)

print(
    f"Observable night for {example_dayObs} from {TWILIGHT_ALT:g} deg (evening) to {TWILIGHT_ALT:g} deg (morning) \n"
    f"runs from {example_observable_night[0].isot} to {example_observable_night[1].isot} TAI"
)

In [ ]:
# Example: length of the observable night for every night of the year.
import matplotlib.pyplot as plt

# Range of nights shown below, as dayObs (YYYYMMDD, inclusive): one full year.
start_observable_night = 20260629
end_observable_night = 20270628


def day_obs_range(first_day_obs, last_day_obs):
    """All dayObs (as YYYYMMDD integers) from first_day_obs to last_day_obs, inclusive."""
    days = pd.date_range(
        pd.to_datetime(str(first_day_obs), format="%Y%m%d"),
        pd.to_datetime(str(last_day_obs), format="%Y%m%d"),
    )
    return [int(d.strftime("%Y%m%d")) for d in days]


def observable_night_table(day_obs_list):
    """Observable night of each dayObs: start and end (TAI MJD) and length (hours)."""
    rows = []
    for day_obs in day_obs_list:
        start, end = observable_night(day_obs)
        rows.append({"day_obs": day_obs, "start": start.tai.mjd, "end": end.tai.mjd})
    nights = pd.DataFrame(rows).set_index("day_obs")
    nights["night_hours"] = (nights.end - nights.start) * 24
    return nights


observable_nights = observable_night_table(
    day_obs_range(start_observable_night, end_observable_night)
)

x = pd.to_datetime(observable_nights.index.astype(str), format="%Y%m%d")
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(x, observable_nights.night_hours)
ax.set_ylabel("Observable night length (hours)")
ax.set_title(
    f"Observable night length, dayObs {start_observable_night} to {end_observable_night} "
    f"(Sun {TWILIGHT_ALT:g} deg to {TWILIGHT_ALT:g} deg)"
)
ax.grid(alpha=0.3)
fig.autofmt_xdate()

In [ ]:
# Example: calendar of the observable night length. One row per week (Monday to Sunday), one cell per night.
# The two helpers below are reused by the calendar of the historical comparison in part 2.
from matplotlib.colors import ListedColormap, Normalize

BLUES = [
    "#cde2fb",
    "#9ec5f4",
    "#6da7ec",
    "#3987e5",
    "#256abf",
    "#184f95",
    "#0d366b",
]  # sequential, one hue


def calendar_grid(hours):
    """Weeks x 7 array (Monday to Sunday) of a per-night value (Series indexed by dayObs), and the first Monday."""
    days = pd.to_datetime(hours.index.astype(str), format="%Y%m%d")
    first_monday = days.min() - pd.Timedelta(days=days.min().weekday())
    nweeks = (days.max() - first_monday).days // 7 + 1
    cal = np.full((nweeks, 7), np.nan)
    for day, h in zip(days, hours.values):
        cal[(day - first_monday).days // 7, day.weekday()] = h
    return cal, first_monday


def draw_calendar(
    ax,
    hours,
    cmap,
    norm,
    title,
    label=lambda h: f"{h:.0f}",
    ytick_every=4,
    fontsize=6.5,
):
    """Draw a calendar of a per-night value on `ax`, with the value written in every cell."""
    cal, first_monday = calendar_grid(hours)
    nweeks = cal.shape[0]
    im = ax.imshow(np.ma.masked_invalid(cal), cmap=cmap, norm=norm, aspect="auto")
    mid = 0.5 * (norm.vmin + norm.vmax)
    for (i, j), h in np.ndenumerate(cal):
        if not np.isnan(h):
            ax.text(
                j,
                i,
                label(h),
                ha="center",
                va="center",
                fontsize=fontsize,
                color="white" if h > mid else "#0b0b0b",
            )
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=8)
    ax.xaxis.tick_top()
    ticks = range(0, nweeks, ytick_every)
    ax.set_yticks(ticks)
    ax.set_yticklabels(
        [(first_monday + pd.Timedelta(weeks=int(i))).strftime("%b %d") for i in ticks],
        fontsize=8,
    )
    ax.set_xticks(np.arange(-0.5, 7, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nweeks, 1), minor=True)
    ax.grid(which="minor", color="white", lw=1.5)
    ax.tick_params(which="both", length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, loc="left", fontsize=10, pad=24)
    return im


hours = observable_nights.night_hours
norm = Normalize(np.floor(hours.min()), np.ceil(hours.max()))
fig, ax = plt.subplots(figsize=(6, 14))
im = draw_calendar(
    ax,
    hours,
    ListedColormap(BLUES),
    norm,
    "Observable night length (hours)",
    label=lambda h: f"{h:.1f}",
    ytick_every=4,
)
fig.colorbar(
    im, ax=ax, orientation="horizontal", fraction=0.025, pad=0.04, aspect=30
).set_label("Hours")

# Part 2: historical comparison of other definitions (NOT VALID)

> **Everything below this point is kept for historical purposes only.** The definitions compared here, other than definition 1 (the full night, which *is* the definition above), are **not valid ways to calculate the observable night** and must not be used for system availability.

While choosing the definition, the full night between -12 and -12 deg twilight was compared with versions that also remove time using the observatory state recorded in the EFD (same source as `notebooks/efficiency.ipynb`):

1. Full night, -12 to -12 deg twilight (the adopted definition, part 1).
2. Full night minus `DOWNTIME` states. *Not valid.*
3. Full night minus `WEATHER` states. *Not valid.*
4. Full night minus `DOWNTIME` and `WEATHER` states. *Not valid.*

The argument for not removing weather time: observing is not limited to an open dome, since closed-dome (calibration) observations can be taken during weather closures, so `WEATHER` time should not be removed from the observable night. The comparison measures how much `OPERATIONAL` time falls inside `WEATHER` periods.

This part needs access to the EFD (USDF) and the cells of part 1 to have been run. The window of nights compared is set in the next cell and is independent of the year shown in part 1.

In [ ]:
# Part 2 setup: window of nights to compare, and access to the EFD (not needed for part 1).
import os

from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.observatory_status as observatory_status

history_start_dayobs = 20260501  # First dayObs compared (inclusive)
history_end_dayobs = 20261016  # Last dayObs compared (inclusive)

day_obs_list = day_obs_range(history_start_dayobs, history_end_dayobs)
t_start = rn_dayobs.day_obs_to_time(history_start_dayobs)
t_end = rn_dayobs.day_obs_to_time(history_end_dayobs) + TimeDelta(1, format="jd")
print(
    f"historical comparison: dayObs {history_start_dayobs} to {history_end_dayobs}, {len(day_obs_list)} nights"
)

# Connections (EFD for dome + observatory state); token handling as in efficiency.ipynb.
if os.getenv("EXTERNAL_INSTANCE_URL", "") != "":
    tokenfile = None
    site = None
endpoints = connections.get_clients()

In [ ]:
# Observable night of every night in the comparison window, from the definition in part 1.
# The dome-open table is not a measure here: it is only an input to count_observatory_states.
dome_open = observatory_status.get_dome_open_close(t_start, t_end, endpoints["efd"])

night_times = observable_night_table(day_obs_list)
display(night_times[["night_hours"]].round(2).describe())

## EFD observatory states

`get_observatory_state_times` returns the observatory's own state periods from the EFD, and `count_observatory_states` reduces them to per-night hours (the same approach as `notebooks/efficiency.ipynb`). The state columns are:

`operational_hours`, `fault_down`, `idle_down`, `weather_down`, `downtime_down`, `unknown_down`.

**These state periods overlap and must not be summed.** A WEATHER period typically spans the whole night while OPERATIONAL / IDLE / FAULT periods run inside it, so the columns can total well over `night_hours`. `count_observatory_states` only resolves FAULT during DOWNTIME (via `contributed_hours`). Treat each column as "hours during which this was true", not as a partition of the night.

**Nights with no status messages.** The EFD logs `observatoryStatus` only when the state changes, and `get_observatory_state_times` returns periods only for nights that have at least one message. A night with none (e.g. while the observatory sits in DOWNTIME) would silently disappear. `fill_missing_nights` carries the state in force at the end of the previous night forward over those nights. Nights before the first message in the window cannot be filled and are excluded.

**Overrides.** `NIGHT_STATE_OVERRIDES` replaces the logged state of individual nights where it is known to be wrong; it is applied only to the nights listed, never as a general rule. Currently `20260925`, logged as `UNKNOWN`, is set to `DOWNTIME`. The overridden state is what is carried forward, so the quiet nights 20260926-20260928 are also `DOWNTIME`.

In [ ]:
# Hours during which each state was true (these overlap - see markdown above).
state_cols = [
    "operational_hours",
    "fault_down",
    "idle_down",
    "weather_down",
    "downtime_down",
    "unknown_down",
]

# Manual overrides of the state for a whole night, as {dayObs: types in force all night}. Use only
# where the logged state is known to be wrong; it is applied to the nights listed, never as a rule.
# A bare DOWNTIME is shown by the library as IDLE | DOWNTIME, so both types are given.
# 20260925 is logged as UNKNOWN but the observatory was in DOWNTIME. Because the overridden state is
# also what gets carried forward, the quiet nights 20260926-20260928 (no status messages until
# DOWNTIME is logged on 20260929) are filled as DOWNTIME too.
NIGHT_STATE_OVERRIDES = {20260925: ["DOWNTIME", "IDLE"]}


def override_night_states(periods, overrides):
    """Replace all periods of the listed nights by one full-night period per type."""
    for d, types in overrides.items():
        night = periods[periods.day_obs == d]
        if len(night) == 0:
            print(f"override for {d} ignored: no periods that night")
            continue
        was = sorted(night.type.unique())
        sunset12, sunrise12 = night.sunset12.iloc[0], night.sunrise12.iloc[0]
        hours = (sunrise12 - sunset12).total_seconds() / 3600
        new = pd.DataFrame(
            [
                {
                    "day_obs": d,
                    "sunset12": sunset12,
                    "sunrise12": sunrise12,
                    "start": sunset12,
                    "end": sunrise12,
                    "hours": hours,
                    "type": t,
                }
                for t in types
            ]
        )
        periods = pd.concat([periods[periods.day_obs != d], new], ignore_index=True)
        print(f"override: {d} set to {types} (logged: {was})")
    return periods.sort_values(["day_obs", "start"], ignore_index=True)


def fill_missing_nights(periods, nights, tol_s=1.0):
    """Carry the last known state forward across nights with no EFD status messages.

    observatoryStatus is only logged on a change of state, so a night with no messages means the
    state persisted. The library builds periods only for nights that have messages, so those nights
    would otherwise vanish. The types still in force at sunrise of the last night with periods
    (e.g. DOWNTIME, WEATHER) are repeated over each following empty night, clipped to now.
    Returns (periods, filled_nights, unfilled_nights, carried_types).
    """
    have = set(periods.day_obs.unique())
    tnow = Time.now().utc
    last_types, rows, filled, unfilled = None, [], [], []
    for d in sorted(nights):
        if d in have:
            per = periods[periods.day_obs == d]
            in_force = per[(per.sunrise12 - per.end).dt.total_seconds() <= tol_s]
            last_types = sorted(in_force.type.unique())
            continue
        sunset, sunrise = (
            t.utc for t in rn_dayobs.day_obs_sunset_sunrise(d, TWILIGHT_ALT)
        )
        if tnow <= sunset:
            continue  # night has not started yet
        if last_types is None:
            unfilled.append(d)  # nothing earlier in the window to carry forward
            continue
        sunrise = min(sunrise, tnow)
        hours = (sunrise - sunset).jd * 24
        for typ in last_types:
            rows.append(
                {
                    "day_obs": d,
                    "sunset12": sunset.datetime,
                    "sunrise12": sunrise.datetime,
                    "start": sunset.datetime,
                    "end": sunrise.datetime,
                    "hours": hours,
                    "type": typ,
                }
            )
        filled.append(d)
    if rows:
        periods = pd.concat(
            [periods, pd.DataFrame(rows)], ignore_index=True
        ).sort_values(["day_obs", "start"], ignore_index=True)
    return periods, filled, unfilled, last_types


obs_status_periods = observatory_status.get_observatory_state_times(
    t_start, t_end, endpoints["efd"]
)

# Drop columns from a previous run so the cell can be re-run.
night_times = night_times.drop(columns=state_cols, errors="ignore")

if len(obs_status_periods) > 0:
    obs_status_periods = override_night_states(
        obs_status_periods, NIGHT_STATE_OVERRIDES
    )
    obs_status_periods, filled_nights, unfilled_nights, carried_types = (
        fill_missing_nights(obs_status_periods, day_obs_list)
    )
    if filled_nights:
        print(
            f"{len(filled_nights)} nights with no EFD status messages carried forward from the "
            f"previous known state (last carried: {carried_types}): {filled_nights[0]} to {filled_nights[-1]}"
        )
    if unfilled_nights:
        print(
            f"{len(unfilled_nights)} nights before the first status message left empty: {unfilled_nights}"
        )
    # count_observatory_states adds `contributed_hours`, which discounts FAULT
    # periods occurring during DOWNTIME, and returns per-day_obs sums.
    obs_status_periods, state_summary = observatory_status.count_observatory_states(
        obs_status_periods, dome_open
    )
    # state_summary also carries sunset12/sunrise12/night_hours and dome-open columns, which
    # night_times does not need - take only the state columns to avoid collisions.
    night_times = night_times.join(state_summary[state_cols], how="left")
else:
    print("No observatory state periods found in the EFD for this range.")
    filled_nights = []
    for c in state_cols:
        night_times[c] = np.nan

display(night_times[["night_hours"] + state_cols].round(2))

In [ ]:
# Per-night quantities used by the denominators and numerators below.
# Nights without EFD state data (including a night that has not started yet) are dropped
# rather than treated as zero. Carried-forward nights have periods, so they are kept.
has_states = night_times[state_cols].notna().all(axis=1) & night_times.index.isin(
    obs_status_periods.day_obs.unique()
)
print(f"{int(has_states.sum())} of {len(night_times)} nights have EFD state data")
p = night_times[has_states].copy()
# Weather/downtime can overlap each other, so clip the subtractions at zero.
p["window_hours"] = p.night_hours
p["WEATHER"] = p.weather_down.clip(upper=p.night_hours)
p["DOWNTIME"] = p.downtime_down.clip(upper=p.night_hours)
p["OPERATIONAL"] = p.operational_hours
p["IDLE"] = p.idle_down
p["FAULT"] = p.fault_down
p["UNKNOWN"] = p.unknown_down
labels = ["WEATHER", "DOWNTIME", "FAULT", "IDLE", "OPERATIONAL", "UNKNOWN"]
display(p[["window_hours"] + labels].round(2).head(10))

## Historical definitions (only definition 1 is valid)

For the night window `W` (`night_hours`, Sun -12 deg to -12 deg, from part 1), `U(types)` is the length of the **union** of all EFD state periods of those types, clipped to `W`:

| Definition | Observable hours | Status |
|---|---|---|
| `1_full_night` | `W` | **adopted** (part 1) |
| `2_minus_downtime` | `W - U(DOWNTIME)` | not valid |
| `3_minus_weather` | `W - U(WEATHER)` | not valid |
| `4_minus_downtime_weather` | `W - U(DOWNTIME, WEATHER)` | not valid |

`UNKNOWN` time is disregarded: it is not removed in any definition.

The state periods overlap (WEATHER can span the whole night with OPERATIONAL / IDLE / FAULT inside it, and WEATHER and DOWNTIME can coincide), so taking the union rather than summing per-state hours avoids double-subtracting. For each definition we also report `operational_excluded`: the hours of `OPERATIONAL` state that fall inside the *excluded* intervals. A non-zero value means the definition throws away time in which the observatory was recorded as operational.

In [ ]:
EXCLUDE = {
    "1_full_night": [],
    "2_minus_downtime": ["DOWNTIME"],
    "3_minus_weather": ["WEATHER"],
    "4_minus_downtime_weather": ["DOWNTIME", "WEATHER"],
}
defs = list(EXCLUDE)


def definition_label(name):
    """Readable name of a definition, worded as in the notebook introduction (used by the plots)."""
    number = defs.index(name) + 1
    types = EXCLUDE[name]
    if not types:
        return f"{number}. Full night"
    joined = (
        types[0] if len(types) == 1 else ", ".join(types[:-1]) + " and " + types[-1]
    )
    return f"{number}. Full night minus {joined}"


def union_hours(periods, types, lo, hi):
    """Hours covered by the union of `types` periods, clipped to [lo, hi]."""
    d = periods[periods.type.isin(types)].sort_values("start")
    total, cur_end = 0.0, lo
    for s, e in zip(d.start, d.end):
        s, e = max(s, lo, cur_end), min(e, hi)
        if e > s:
            total += (e - s).total_seconds() / 3600
            cur_end = e
    return total


windows = obs_status_periods.groupby("day_obs")[["sunset12", "sunrise12"]].first()
rows = []
for day_obs, w in windows.iterrows():
    per = obs_status_periods[obs_status_periods.day_obs == day_obs]
    lo, hi = w.sunset12, w.sunrise12
    night = (hi - lo).total_seconds() / 3600
    r = {"day_obs": day_obs, "night_hours": night}
    u_op = union_hours(per, ["OPERATIONAL"], lo, hi)
    for name, types in EXCLUDE.items():
        u_ex = union_hours(per, types, lo, hi) if types else 0.0
        r[name] = night - u_ex
        # OPERATIONAL time inside the excluded intervals = U(op) + U(ex) - U(op or ex)
        r[f"operational_excluded[{name}]"] = (
            u_op + u_ex - union_hours(per, ["OPERATIONAL"] + types, lo, hi)
            if types
            else 0.0
        )
    rows.append(r)
obs_night = pd.DataFrame(rows).set_index("day_obs")

display(obs_night[["night_hours"] + defs].round(2))

summary = pd.DataFrame(
    {
        "pooled_hours": obs_night[defs].sum(),
        "pct_of_full_night": obs_night[defs].sum()
        / obs_night["1_full_night"].sum()
        * 100,
        "mean_per_night": obs_night[defs].mean(),
        "nights_with_zero_hours": (obs_night[defs] < 1e-6).sum(),
        "operational_excluded_hours": [
            obs_night[f"operational_excluded[{d}]"].sum() for d in defs
        ],
    }
)
display(summary.round(2))
print(
    f"{len(obs_night)} nights; OPERATIONAL hours total (union): "
    f"{sum(union_hours(obs_status_periods[obs_status_periods.day_obs == d], ['OPERATIONAL'], w.sunset12, w.sunrise12) for d, w in windows.iterrows()):.1f}"
)

## Visualizations of the historical comparison

Every figure below is titled as historical. Definitions 2 to 4 are not valid.

1. EFD state per night as a stacked bar. The raw states overlap, so each night is split into exclusive pieces that sum to the night length (priority: `DOWNTIME`, then `WEATHER` split into time recorded `OPERATIONAL` and the rest, then `OPERATIONAL`, `FAULT`, `UNKNOWN`, `IDLE`). A strip below the bars marks carried-forward nights.
2. *Diagnostic only:* a night-by-time raster of the same states (one column per night, time since evening twilight on the y axis), to see when in the night each state occurred.
3. Calendar of observable hours per night under each definition (one row per week, one panel per definition, common colour scale; the title gives the pooled hours).
4. Observable hours per night under each definition, as lines.

In [ ]:
# 1. EFD state per night as a stacked bar. HISTORICAL: part of the comparison of definitions that are not valid.
# The raw states overlap, so each night is split into
# mutually exclusive pieces that sum to the night length, using this priority (first match wins):
# DOWNTIME > WEATHER (split into time recorded OPERATIONAL and the rest) > OPERATIONAL > FAULT >
# UNKNOWN > IDLE. DOWNTIME overrides the other states, as in count_observatory_states.
# Time before the first status message of the window has no state; it is shown as "no state recorded".
HISTORICAL_TITLE = "HISTORICAL comparison (definitions 2-4 are not valid definitions of the observable night)"

STACK = [  # bottom to top: (label, colour); order and colours checked for adjacent-pair CVD separation
    ("OPERATIONAL", "#008300"),
    ("WEATHER, recorded OPERATIONAL", "#2a78d6"),
    ("WEATHER", "#2a78d6"),
    ("IDLE", "#eda100"),
    ("FAULT", "#e34948"),
    ("DOWNTIME", "#6250d6"),
    ("UNKNOWN", "#898781"),
    ("no state recorded", "#e1e0d9"),
]
LABEL = {label: i for i, (label, _) in enumerate(STACK)}


def night_state_codes(per, lo, hi):
    """STACK index of the state in force in each 1-minute bin of one night, by the priority above."""
    nbins = int(round((hi - lo).total_seconds() / 60))
    t0, t1 = lo.to_datetime64().astype("int64"), hi.to_datetime64().astype("int64")
    mid = t0 + (np.arange(nbins) + 0.5) * (t1 - t0) / nbins

    def mask(kind):
        m = np.zeros(nbins, bool)
        d = per[per.type == kind]
        for s, e in zip(d.start, d.end):
            m |= (mid >= s.to_datetime64().astype("int64")) & (
                mid < e.to_datetime64().astype("int64")
            )
        return m

    idle, unk, fault, op, wx, down = (
        mask(k)
        for k in ["IDLE", "UNKNOWN", "FAULT", "OPERATIONAL", "WEATHER", "DOWNTIME"]
    )
    codes = np.full(nbins, LABEL["no state recorded"], dtype=np.int8)
    # later assignments win: IDLE < UNKNOWN < FAULT < OPERATIONAL < WEATHER < WEATHER+OPERATIONAL < DOWNTIME
    codes[idle] = LABEL["IDLE"]
    codes[unk] = LABEL["UNKNOWN"]
    codes[fault] = LABEL["FAULT"]
    codes[op] = LABEL["OPERATIONAL"]
    codes[wx] = LABEL["WEATHER"]
    codes[wx & op] = LABEL["WEATHER, recorded OPERATIONAL"]
    codes[down] = LABEL["DOWNTIME"]
    return codes


state_windows = obs_status_periods.groupby("day_obs")[["sunset12", "sunrise12"]].first()
state_codes = {
    d: night_state_codes(
        obs_status_periods[obs_status_periods.day_obs == d], w.sunset12, w.sunrise12
    )
    for d, w in state_windows.iterrows()
}
night_len = {
    d: (w.sunrise12 - w.sunset12).total_seconds() / 3600
    for d, w in state_windows.iterrows()
}
part = (
    pd.DataFrame(
        {
            d: np.bincount(c, minlength=len(STACK)) * night_len[d] / len(c)
            for d, c in state_codes.items()
        },
        index=[s[0] for s in STACK],
    )
    .T.reindex(p.index)
    .fillna(0)
)
print(
    f"{len(part)} nights; largest |sum - night length|: {(part.sum(axis=1) - p.window_hours).abs().max():.3f} h; "
    f"nights with >0.05 h of no state recorded: {int((part['no state recorded'] > 0.05).sum())}"
)

xs = pd.to_datetime(part.index.astype(str), format="%Y%m%d")
fig, ax = plt.subplots(figsize=(16, 5))
bottom = np.zeros(len(part))
handles = []
for label, colour in STACK:
    style = dict(edgecolor="white", lw=0.3)
    if (
        label == "WEATHER, recorded OPERATIONAL"
    ):  # blue with green texture: operational while weather-flagged
        style = dict(edgecolor="#008300", lw=0, hatch="////")
    bars = ax.bar(xs, part[label], bottom=bottom, width=1.0, color=colour, **style)
    handles.append(bars)
    bottom += part[label].to_numpy()
# strip under the bars marking nights with no status messages (state carried forward)
strip = ax.bar(
    pd.to_datetime([str(d) for d in filled_nights], format="%Y%m%d"),
    0.5,
    bottom=-0.7,
    width=1.0,
    color="#c3c2b7",
    lw=0,
)
ax.legend(
    [h[0] for h in handles] + [strip[0]],
    [s[0] for s in STACK] + ["carried forward (no status messages)"],
    ncol=5,
    loc="lower left",
    bbox_to_anchor=(0, 1.0),
    fontsize=8,
    frameon=False,
)
ax.set_ylabel("Hours of the night")
ax.set_ylim(-0.9, None)
ax.set_yticks(np.arange(0, 13, 2))
ax.grid(alpha=0.2, axis="y")
ax.set_axisbelow(True)
ax.set_title(HISTORICAL_TITLE, loc="left", fontsize=10, color="#b00020", pad=40)
fig.autofmt_xdate()

In [ ]:
# 2. HISTORICAL, DIAGNOSTIC: night-by-time raster. One column per night, hours since the
# evening -12 deg twilight on the y axis, coloured by state, so you can see when in the night each state occurred.
# Uses state_codes from the stacked-bar cell above.
import matplotlib.dates as mdates
from matplotlib.patches import Patch

BIN_H = 1 / 60  # one-minute bins
NROWS = int(np.ceil(max(night_len.values()) * 60))
dates = pd.date_range(
    pd.to_datetime(str(part.index.min()), format="%Y%m%d"),
    pd.to_datetime(str(part.index.max()), format="%Y%m%d"),
)
grid = np.full(
    (NROWS, len(dates)), np.nan
)  # NaN: no such night, or below the end of a shorter night
for j, day in enumerate(dates):
    key = int(day.strftime("%Y%m%d"))
    if key in state_codes and key in part.index:
        grid[: len(state_codes[key]), j] = state_codes[key]

fig, ax = plt.subplots(figsize=(16, 6.5))
x0, x1 = mdates.date2num(dates[0]) - 0.5, mdates.date2num(dates[-1]) + 0.5
cmap = ListedColormap([c for _, c in STACK])
cmap.set_bad("white")
ax.imshow(
    np.ma.masked_invalid(grid),
    cmap=cmap,
    vmin=-0.5,
    vmax=len(STACK) - 0.5,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=[x0, x1, 0, NROWS * BIN_H],
)
# green texture where weather-flagged time was recorded OPERATIONAL
xc = mdates.date2num(dates)
yc = (np.arange(NROWS) + 0.5) * BIN_H
with plt.rc_context({"hatch.color": "#008300", "hatch.linewidth": 0.8}):
    ax.contourf(
        xc,
        yc,
        (grid == LABEL["WEATHER, recorded OPERATIONAL"]).astype(float),
        levels=[0.5, 1.5],
        colors="none",
        hatches=["////"],
    )
ax.bar(
    xc[[d in filled_nights for d in dates.strftime("%Y%m%d").astype(int)]],
    0.4,
    bottom=-0.6,
    width=1.0,
    color="#c3c2b7",
    lw=0,
)  # strip: nights with no status messages, state carried forward
ax.xaxis_date()
ax.set_xlim(x0, x1)
ax.set_ylim(-0.7, NROWS * BIN_H)
ax.set_ylabel("Hours after evening -12 deg twilight")
ax.set_title(HISTORICAL_TITLE, loc="left", fontsize=10, color="#b00020")
patches = [
    Patch(
        facecolor=c,
        edgecolor="#008300" if l == "WEATHER, recorded OPERATIONAL" else c,
        hatch="////" if l == "WEATHER, recorded OPERATIONAL" else None,
        label=l,
    )
    for l, c in STACK
]
patches.append(Patch(facecolor="#c3c2b7", label="carried forward (no status messages)"))
ax.legend(
    handles=patches,
    ncol=5,
    loc="upper left",
    bbox_to_anchor=(0, -0.14),
    fontsize=8,
    frameon=False,
)
fig.autofmt_xdate()

In [ ]:
# 3. HISTORICAL: calendar of observable hours per night under each definition, on one colour scale,
# to compare the options night by night. One row per week (Monday to Sunday); the title gives the pooled hours.
# Uses state_codes (per-minute state of every night) from the stacked-bar cell, and draw_calendar from part 1.
type_codes = {
    "DOWNTIME": ["DOWNTIME"],
    "WEATHER": ["WEATHER", "WEATHER, recorded OPERATIONAL"],
}


def observable_hours(name):
    """Observable hours of each night under definition `name` (a key of EXCLUDE)."""
    excluded = [LABEL[l] for t in EXCLUDE[name] for l in type_codes.get(t, [])]
    return pd.Series(
        {
            d: (~np.isin(state_codes[d], excluded)).sum()
            * night_len[d]
            / len(state_codes[d])
            for d in part.index
        }
    )


def cell_label(h):
    return "0" if h < 0.05 else "<1" if h < 0.5 else f"{h:.0f}"


cal_hours = {name: observable_hours(name) for name in EXCLUDE}
total_night = sum(night_len[d] for d in part.index)

cal_cmap = ListedColormap(BLUES)
cal_cmap.set_under("#e1e0d9")  # no observable time
cal_cmap.set_bad("white")  # not a night in the window

fig, axes = plt.subplots(
    1, len(cal_hours), figsize=(5 * len(cal_hours), 9), sharey=True
)
for cax, (name, hours) in zip(np.atleast_1d(axes), cal_hours.items()):
    im = draw_calendar(
        cax,
        hours,
        cal_cmap,
        Normalize(0.05, 12),
        f"{definition_label(name)}\n{hours.sum():.0f} h observable ({hours.sum() / total_night * 100:.0f}% of the night)",
        label=cell_label,
        ytick_every=2,
    )
fig.suptitle(HISTORICAL_TITLE, x=0.01, ha="left", fontsize=10, color="#b00020")
cb = fig.colorbar(
    im,
    ax=axes,
    orientation="horizontal",
    fraction=0.025,
    pad=0.06,
    extend="min",
    aspect=40,
)
cb.set_label("Observable hours per night (light grey: none)", fontsize=8)

In [ ]:
# 4. HISTORICAL: observable hours per night under each definition (pooled hours are in the summary table above)
def_colors = dict(zip(defs, ["k", "tab:purple", "tab:orange", "tab:blue"]))
fig, ax = plt.subplots(figsize=(12, 4.8))

xd = pd.to_datetime(obs_night.index.astype(str), format="%Y%m%d")
for k, dname in enumerate(defs):
    # small x offset so coincident definitions stay visible
    ax.plot(
        xd + pd.Timedelta(hours=3 * k),
        obs_night[dname],
        color=def_colors[dname],
        marker=".",
        lw=1,
        label=definition_label(dname),
    )
ax.set_ylabel("Observable hours per night")
ax.set_title(
    HISTORICAL_TITLE
    + "\nObservable night under each definition (x offset for visibility)",
    loc="left",
    fontsize=10,
)
ax.legend(loc="center right", fontsize=8)
ax.grid(alpha=0.3)
fig.autofmt_xdate()

## Notes on the historical comparison

The definition of the observable night is the one in part 1 (Sun -12 deg to -12 deg). These notes only qualify the historical comparison in part 2.

- **Evidence against removing `WEATHER`**: definitions 3 and 4 remove `WEATHER` time, but some of that time is recorded as `OPERATIONAL` (see `operational_excluded` above). Closed-dome (calibration) observations during weather closures are consistent with this, although the state record alone does not show that they happened. Counting calibration exposures taken during `WEATHER` periods (e.g. from the Consolidated Database or the exposure log) was not done.
- **Nights with no status messages are carried forward.** `observatoryStatus` is logged only on a change of state and the library builds periods only for nights that have a message, so a quiet night would otherwise drop out of the statistics. `fill_missing_nights` repeats the state in force at the end of the previous night (e.g. `DOWNTIME`) over each empty night. This is an assumption: a long gap with no events could also mean a gap in the EFD record rather than a persistent state. Nights before the first message in the window cannot be filled and are left out.
- **`UNKNOWN`** is not removed in any definition. It is negligible in the first weeks of the window but accounts for tens of hours over a longer one.
- **`DOWNTIME` and `WEATHER` coincide on some nights** (e.g. 20260715, 20260716), which is why the excluded time is the union of the periods.
- **Weather data source**: only the EFD `WEATHER` state was used. It was not compared with the narrative-log weather time lost or a weather-station criterion (wind, humidity from the EFD).
- **Choice of date window**: with carry-forward, long periods of carried `DOWNTIME` dominate pooled totals, so the pooled figures depend strongly on the window set in the part 2 setup cell.